# Tea Leaf Quality Prediction - Updated Model Training (Colab)

Trains two XGBoost models using the **8 raw quality parameters** as features:

1. **XGBClassifier** - Predicts quality grade label (Premium, Superior, High, Good, Standard, Commercial, Low, Reject)
2. **XGBRegressor** - Predicts quality score percentage (0-100)

### Features (9 total)
- Tea Flavor (label encoded)
- Particle Size (mesh)
- Moisture (%)
- Color Value (L*)
- Aroma Power (/10)
- Taste Strength (/10)
- Solubility (%)
- Caffeine (%)
- Fineness (%)

### Targets
- `Grade Label` (8 classes)
- `Quality Score` (continuous 0-100)

## 0. Install Dependencies & Upload Dataset

In [ ]:
!pip install xgboost scikit-learn pandas openpyxl matplotlib seaborn -q

In [ ]:
from google.colab import files
print('Upload Tea_Quality_Dataset_V2.xlsx file:')
uploaded = files.upload()

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score, confusion_matrix
from xgboost import XGBClassifier, XGBRegressor
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [ ]:
dataset_path = 'Tea_Quality_Dataset_V2.xlsx'
print(f'Loading dataset from: {dataset_path}')
df = pd.read_excel(dataset_path)
print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nGrade Label distribution:')
print(df['Grade Label'].value_counts().sort_index())
df.head()

## 2. Define Features and Targets

In [ ]:
feature_cols = [
    'Tea Flavor',
    'Particle Size',
    'Moisture (%)',
    'Color Value',
    'Aroma Power',
    'Taste Strength',
    'Solubility (%)',
    'Caffeine (%)',
    'Fineness (%)'
]

target_class = 'Grade Label'
target_reg = 'Quality Score'

X = df[feature_cols].copy()
y_class = df[target_class].copy()
y_reg = df[target_reg].copy()

print(f'Features shape: {X.shape}')
print(f'Class target unique values: {y_class.unique().tolist()}')
print(f'Regression target range: {y_reg.min():.1f} - {y_reg.max():.1f}')

## 3. Encode and Scale

In [ ]:
# Encode Tea Flavor (categorical -> numeric)
tea_flavor_encoder = LabelEncoder()
X['Tea Flavor'] = tea_flavor_encoder.fit_transform(X['Tea Flavor'])
print(f'Tea Flavor classes: {tea_flavor_encoder.classes_.tolist()}')

# Encode Grade Label target
quality_encoder = LabelEncoder()
y_class_encoded = quality_encoder.fit_transform(y_class)
print(f'Grade Label classes: {quality_encoder.classes_.tolist()}')

# Scale all features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled features shape: {X_scaled.shape}')

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X_scaled, y_class_encoded, y_reg,
    test_size=0.2, random_state=42, stratify=y_class_encoded
)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## 5. Train XGBoost Classifier (8 Grade Labels)

In [ ]:
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)
clf.fit(X_train, y_cls_train)

y_cls_pred = clf.predict(X_test)
acc = accuracy_score(y_cls_test, y_cls_pred)
print(f'Classification Accuracy: {acc:.4f}')
print()
print(classification_report(y_cls_test, y_cls_pred, target_names=quality_encoder.classes_.tolist()))

### Classification Metrics & Charts

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_cls_test, y_cls_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=quality_encoder.classes_, 
            yticklabels=quality_encoder.classes_)
plt.title('Grade Classification - Confusion Matrix', fontsize=16)
plt.ylabel('Actual Grade')
plt.xlabel('Predicted Grade')
plt.tight_layout()
plt.show()

# Feature Importance for Classifier
importance_clf = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': clf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_clf, palette='viridis')
plt.title('Classifier Feature Importance', fontsize=16)
plt.tight_layout()
plt.show()

## 6. Train XGBoost Regressor (Quality Score %)

In [ ]:
reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse'
)
reg.fit(X_train, y_reg_train)

y_reg_pred = reg.predict(X_test)
mae = mean_absolute_error(y_reg_test, y_reg_pred)
r2 = r2_score(y_reg_test, y_reg_pred)
print(f'Regression MAE: {mae:.4f}')
print(f'Regression R2:  {r2:.4f}')

### Regression Metrics & Charts

In [ ]:
# Actual vs Predicted Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.5, color='purple')
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.title('Quality Score Regression: Actual vs Predicted', fontsize=16)
plt.xlabel('Actual Score (%)')
plt.ylabel('Predicted Score (%)')
plt.tight_layout()
plt.show()

# Feature Importance for Regressor
importance_reg = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': reg.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_reg, palette='plasma')
plt.title('Regressor Feature Importance', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Save Models, Encoders, and Metadata

In [ ]:
save_dir = 'quality_models'
os.makedirs(save_dir, exist_ok=True)

# Save XGBoost models
clf.save_model(os.path.join(save_dir, 'quality_classifier.json'))
reg.save_model(os.path.join(save_dir, 'quality_regressor.json'))

# Save encoders and scaler
with open(os.path.join(save_dir, 'tea_flavor_encoder.pkl'), 'wb') as f:
    pickle.dump(tea_flavor_encoder, f)
with open(os.path.join(save_dir, 'quality_encoder.pkl'), 'wb') as f:
    pickle.dump(quality_encoder, f)
with open(os.path.join(save_dir, 'feature_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# Save metadata
metadata = {
    'feature_columns': feature_cols,
    'tea_flavors': tea_flavor_encoder.classes_.tolist(),
    'quality_classes': quality_encoder.classes_.tolist(),
    'classification_accuracy': float(acc),
    'regression_mae': float(mae),
    'regression_r2': float(r2),
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'grade_multipliers': {
        'Premium': 1.35,
        'Superior': 1.25,
        'High': 1.15,
        'Good': 1.05,
        'Standard': 1.00,
        'Commercial': 0.90,
        'Low': 0.75,
        'Reject': 0.50
    }
}
with open(os.path.join(save_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('All models and metadata saved successfully!')
print(f'Files in {save_dir}:', os.listdir(save_dir))

## 8. Download Trained Models

Downloads a ZIP file containing all model files. Extract the contents into:

`backend/ml-model/quality_models/`

In [ ]:
import shutil
from google.colab import files

# Zip the quality_models folder
shutil.make_archive('quality_models', 'zip', '.', 'quality_models')
print('Created quality_models.zip')
print('Contents:', os.listdir(save_dir))

# Download the zip
files.download('quality_models.zip')